# L4 RAG Modern

这个版本保留原来“从商品目录里检索并回答问题”的功能，但做了两点更新：

- 不再依赖旧的 `VectorstoreIndexCreator` / `RetrievalQA`
- 不依赖你本地缺失的 `langchain_huggingface` 包，而是直接用本地 `SentenceTransformer` 模型实现 Embeddings

这样 notebook 更贴近 LangChain 1.x 的推荐思路：`Retriever + Prompt + LLM` 显式组合。


In [1]:
import os    # 导入操作系统接口模块，用于处理环境变量和系统路径
from pathlib import Path  # 导入面向对象的路径操作库，比 os.path 更简洁易用

from dotenv import load_dotenv, find_dotenv  # 用于自动搜索并加载 .env 文件中的环境变量（如 API Keys）
from sentence_transformers import SentenceTransformer  # 导入 Hugging Face 的句子转换器，用于本地生成向量

from langchain_core.documents import Document  # 导入 LangChain 的文档对象标准类
from langchain_core.embeddings import Embeddings  # 导入 LangChain 的嵌入模型基类，用于自定义适配
from langchain_core.output_parsers import StrOutputParser  # 导入字符串输出解析器，将模型返回的消息转为纯文本
from langchain_core.prompts import ChatPromptTemplate  # 导入聊天提示词模板工具
from langchain_core.runnables import RunnablePassthrough  # 导入透传组件，用于在链中传递原始输入
from langchain_core.vectorstores import InMemoryVectorStore  # 导入内存向量数据库，适合快速测试和小型应用
from langchain_openai import ChatOpenAI  # 导入 OpenAI 聊天模型适配器
from langchain_community.document_loaders import CSVLoader  # 导入社区提供的 CSV 文件加载器

# 自动寻找并加载 .env 配置文件，通常用于初始化 OPENAI_API_KEY
_ = load_dotenv(find_dotenv())

# --- 路径解析逻辑：用于在特定的项目结构下定位模型 ---

def find_workspace_root() -> Path:
    """
    向上递归寻找名为 'AIAgent' 的文件夹，将其作为项目的根目录。
    这能确保代码在不同层级的子目录下运行时都能准确找到模型文件。
    """
    current = Path.cwd().resolve() # 获取当前工作目录的绝对路径
    for candidate in [current, *current.parents]: # 遍历当前目录及其所有上级目录
        if candidate.name == "AIAgent": # 如果发现目录名匹配
            return candidate
    raise FileNotFoundError("Could not find the AIAgent workspace root.")

def find_local_bge_snapshot() -> Path:
    workspace_root = find_workspace_root()
    snapshot_root = workspace_root / "models" / "models--BAAI--bge-small-en-v1.5" / "snapshots"

    # macOS 下 snapshots 目录里可能混入 .DS_Store 这类隐藏文件。
    # 这里显式只保留“真正存在 config.json 的模型目录”，避免误把隐藏文件当成模型路径。
    snapshots = sorted(
        path for path in snapshot_root.iterdir()
        if path.is_dir() and (path / "config.json").exists()
    )

    if not snapshots:
        raise FileNotFoundError(f"No valid model snapshot found under {snapshot_root}")

    return snapshots[0]

class LocalBGEEmbeddings(Embeddings):
    """
    自定义类，继承自 LangChain 的 Embeddings 基类。
    直接封装 sentence-transformers，避免安装额外的第三方适配包。
    """
    def __init__(self, model_path: str):
        # 初始化时加载本地模型到内存（如果在 M1 Mac 上，它通常会自动利用 Apple Silicon 的优化）
        self.model = SentenceTransformer(model_path)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """
        为多个文档生成嵌入向量。
        normalize_embeddings=True 确保生成的向量长度为 1，方便后续通过余弦相似度进行搜索。
        """
        vectors = self.model.encode(texts, normalize_embeddings=True)
        return vectors.tolist() # 将 numpy 数组转换为标准 Python 列表

    def embed_query(self, text: str) -> list[float]:
        """
        为用户的单条查询问题生成嵌入向量。
        """
        vector = self.model.encode(text, normalize_embeddings=True)
        return vector.tolist()


In [2]:
file = "OutdoorClothingCatalog_1000.csv"
query = (
    "Please list all your shirts with sun protection in a markdown table "
    "and summarize each one."
)

# 1. 先把 CSV 里的每一行商品记录加载成 LangChain Document。
# 后面无论是做向量化、检索，还是把内容拼进 Prompt，操作的基本单位都是 Document。
loader = CSVLoader(file_path=file, encoding="utf-8")
docs = loader.load()

# 2. 定位本地 embedding 模型，并实例化我们上一个代码块里定义的适配器。
# 这一步的本质是：把 sentence-transformers 包装成 LangChain 认识的 Embeddings 接口。
embedding_model_path = str(find_local_bge_snapshot())
embeddings = LocalBGEEmbeddings(embedding_model_path)

# 3. 把原始文档向量化后放进内存向量库。
# from_documents 会在内部完成两件事：
# - 调用 embeddings.embed_documents(...) 生成每条文档的向量
# - 把“原始文本 + 向量”一起保存到向量库中
# 所以这一步其实就是 RAG 的“建立知识索引”。
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)

# 4. 再从向量库导出 retriever。
# retriever 是 RAG 流程里非常关键的角色：
# 用户提问 -> 问题向量化 -> 找相似文档 -> 把文档交给后面的生成链。
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 5. 初始化真正负责“生成答案”的聊天模型。
# 在 RAG 里，LLM 不负责记住整个知识库；
# 它负责阅读 retriever 拿回来的上下文，然后组织成自然语言答案。
llm = ChatOpenAI(
    temperature=0.0,
    model="qwen-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)


In [4]:
def format_docs(retrieved_docs: list[Document]) -> str:
    # 向量检索阶段拿回来的是 Document 对象列表。
    # 但大模型最终只能读取文本，所以这里把多条文档拼成一段 context 字符串。
    # 这就是大多数 RAG 实现里常见的“文档格式化”步骤。
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

# Prompt 在 RAG 里承担两个关键任务：
# 1. 明确告诉模型“只能基于检索上下文回答”
# 2. 规定输入变量名称，让上游链路能稳定注入 context 和 question
rag_prompt = ChatPromptTemplate.from_template(
    '''
    You are a professional outdoor apparel catalog assistant.
    Use only the retrieved context below to answer the question.
    If the context does not contain the answer, say so clearly.

    Context:
    {context}

    Question:
    {question}
    '''
)

# 下面这个 LCEL 表达式就是一条完整的 RAG 链。
# 可以按“数据流”来理解：
#
# {"context": ..., "question": ...}
#   -> 先分别准备 context 和 question
#   -> 填进 rag_prompt
#   -> 交给 llm 生成回答
#   -> 再用 StrOutputParser 把模型输出转成普通字符串
rag_chain = (
    {
        # retriever | format_docs 的意思是：
        # 用户问题 -> retriever 检索 -> 文档列表 -> format_docs 拼成文本
        "context": retriever | format_docs,
        # RunnablePassthrough() 表示原始问题不做处理，直接原样传下去。
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)


In [5]:
# 先单独看一下检索阶段拿回了什么，再看最终回答。
retrieved_docs = retriever.invoke(query)
for index, doc in enumerate(retrieved_docs, start=1):
    print(f"Document {index}")
    print(doc.page_content[:600])
    print("-" * 80)


Document 1
: 709
name: Sunrise Tee
description: Stay cool, comfortable and dry on the hottest days in our women's UV-protective button down shirt. The lightweight, high-performance fabric wicks away moisture and dries quickly.

Size & Fit
Slightly Fitted: Softly shapes the body. Falls at hip.

Why We Love It
Our lightest hot-weather shirt lets you beat the heat. Originally designed for fishing, it's also a great choice for travel thanks to its wrinkle-free fabric and built-in sun protection with a rating of UPF 50+.

Fabric & Care
Lightweight performance synthetic wicks moisture, resists wrinkles and dri
--------------------------------------------------------------------------------
Document 2
: 255
name: Sun Shield Shirt by
description: "Block the sun, not the fun – our high-performance sun shirt is guaranteed to protect from harmful UV rays. 

Size & Fit: Slightly Fitted: Softly shapes the body. Falls at hip.

Fabric & Care: 78% nylon, 22% Lycra Xtra Life fiber. UPF 50+ rated – the 

In [6]:
answer = rag_chain.invoke(query)
print(answer)


Certainly! Here is a markdown table listing all the shirts with sun protection, along with a summary of each one:

| Name                          | Description                                                                 | Size & Fit                                      | Fabric & Care                                                                                         | Additional Features                                                                                                                |
|-------------------------------|-----------------------------------------------------------------------------|-------------------------------------------------|-------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------------------------|
| **Sunrise Tee**               | Stay cool, comfortable, and dry on the hottest days. 